# Pathway Enrichment (Reactome Pathway-Embedding GNN)

In [1]:
import os
import csv
import json
import copy
import pickle
import types
import subprocess
import urllib.request
from collections import defaultdict, namedtuple
from datetime import datetime
from typing import Callable, Optional, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch import Tensor

import dgl
from dgl import DGLGraph
from dgl.data import DGLDataset
from dgl.nn.pytorch import edge_softmax
import dgl.function as fn
from dgl.base import DGLError
from dgl.dataloading import GraphDataLoader

import networkx as nx
from tqdm.auto import tqdm

from sklearn import metrics
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, pairwise_distances
from sklearn.model_selection import train_test_split

from reactome2py import analysis


## `dataset.py` — `PathwayDataset`

DGL dataset that loads pickled NetworkX pathway graphs from `raw/`, converts each to a DGL
graph (encoding `weight` and a binarized `significance` node attribute), and caches the result
in `processed/`.


In [2]:
class PathwayDataset(DGLDataset):
    def __init__(self, root='reactome_embedding/data/emb'):
        raw_dir = os.path.join(root, 'raw')
        save_dir = os.path.join(root, 'processed')
        model_dir = os.path.join(root, 'models')

        os.makedirs(raw_dir, exist_ok=True)
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

        # store them separately if you need to reuse later
        self._raw_dir = raw_dir
        self._save_dir = save_dir
        super().__init__(name='pathway_graph', raw_dir=raw_dir, save_dir=save_dir)
        

    def has_cache(self):
        return len(os.listdir(self.save_dir)) == len(os.listdir(self.raw_dir))

    def __len__(self):
        return len(os.listdir(self.save_dir))

    def __getitem__(self, idx):
        names = sorted(os.listdir(self.save_dir))
        name = names[idx]
        (graph,), _ = dgl.load_graphs(os.path.join(self.save_dir, name))
        return graph, name

    def process(self):
        for cnt, graph_file in enumerate(os.listdir(self.raw_dir)):
            graph_path = os.path.join(self.raw_dir, graph_file)
            nx_graph = pickle.load(open(graph_path, 'rb'))
            for node in nx_graph.nodes:
                if nx_graph.nodes[node]['significance'] == 'significant':
                    nx_graph.nodes[node]['significance'] = 1.0
                else:
                    nx_graph.nodes[node]['significance'] = 0.0
            dgl_graph = dgl.from_networkx(nx_graph, node_attrs=['weight', 'significance'])
            save_path = os.path.join(self.save_dir, f'{graph_file[:-4]}.dgl')
            dgl.save_graphs(save_path, dgl_graph)


## `network.py` — `Network`

Builds a directed NetworkX graph of the full Reactome pathway hierarchy (via the
`ReactomePathwaysRelation.txt` adjacency file and the `eventsHierarchy` JSON API), optionally
annotated with enrichment-analysis weights/significance per pathway.


In [3]:
class Network:
    
    Info = namedtuple('Info', ['name', 'species', 'type', 'diagram'])

    def __init__(self, ea_result=None, kge=None):
        self.txt_url = 'https://reactome.org/download/current/ReactomePathwaysRelation.txt'
        self.json_url = 'https://reactome.org/ContentService/data/eventsHierarchy/9606'

        if kge is not None:
            self.kge = kge
        else:
            self.kge = datetime.now().strftime('%Y-%b-%d-%H-%M')

        self.txt_adjacency = self.parse_txt()
        self.json_adjacency, self.pathway_info = self.parse_json()

        if ea_result is not None:
            self.weights = self.set_weights(ea_result)
        else:
            self.weights = None

        self.name_to_id = self.set_name_to_id()
        self.graph_nx = self.to_networkx()

        self.save_name_to_id()
        self.save_sorted_stids()

    def parse_txt(self):
        txt_adjacency = defaultdict(list)
        found = False

        # Reactome's server 403s the default urllib User-Agent (looks like an
        # unidentified bot); sending a browser-like User-Agent fixes it.
        req = urllib.request.Request(self.txt_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as f:
            lines = f.readlines()

            for line in lines:
                line = line.decode('utf-8')
                stid1, stid2 = line.strip().split()

                if 'R-HSA' not in stid1:
                    if found:
                        break
                    else:
                        continue

                found = True
                txt_adjacency[stid1].append(stid2)

        return dict(txt_adjacency)

    def parse_json(self):
        req = urllib.request.Request(self.json_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as f:
            tree_list = json.load(f)

        json_adjacency = defaultdict(list)
        pathway_info = {}

        for tree in tree_list:
            self.recursive(tree, json_adjacency, pathway_info)

        return dict(json_adjacency), pathway_info

    def recursive(self, tree, json_adjacency, pathway_info):
        id = tree['stId']

        try:
            pathway_info[id] = Network.Info(
                tree['name'],
                tree['species'],
                tree['type'],
                tree.get('diagram', None)
            )
        except KeyError:
            pathway_info[id] = Network.Info(
                tree.get('name', 'NA'),
                tree.get('species', 'NA'),
                tree.get('type', 'NA'),
                None
            )

        children = tree.get('children', [])
        for child in children:
            json_adjacency[id].append(child['stId'])
            self.recursive(child, json_adjacency, pathway_info)

    def set_weights(self, ea_result):
        weights = {}

        for stid in self.pathway_info.keys():
            if stid in ea_result:
                entry = ea_result[stid]
                weights[stid] = {
                    'p_value': entry.get('p_value', 1.0),
                    'significance': entry.get('significance', 'not-found')
                }
            else:
                weights[stid] = {
                    'p_value': 1.0,
                    'significance': 'not-found'
                }

        return weights

    def set_node_attributes(self):
        stids = {}
        names = {}
        weights = {}
        significances = {}

        for stid in self.pathway_info.keys():
            stids[stid] = stid
            names[stid] = self.pathway_info[stid].name

            if self.weights is None:
                weights[stid] = 1.0
                significances[stid] = 'not-found'
            else:
                weights[stid] = self.weights[stid]['p_value']
                significances[stid] = self.weights[stid]['significance']

        return stids, names, weights, significances

    def to_networkx(self, type='json'):
        graph_nx = nx.DiGraph()

        graph = self.json_adjacency if type == 'json' else self.txt_adjacency

        for key, values in graph.items():
            for value in values:
                graph_nx.add_edge(key, value)

        stids, names, weights, significances = self.set_node_attributes()

        nx.set_node_attributes(graph_nx, stids, 'stId')
        nx.set_node_attributes(graph_nx, names, 'name')
        nx.set_node_attributes(graph_nx, weights, 'weight')
        nx.set_node_attributes(graph_nx, significances, 'significance')

        return graph_nx

    def add_significance_by_stid(self, stid_list):
        for stid in stid_list:
            if stid in self.graph_nx.nodes:
                self.graph_nx.nodes[stid]['significance'] = 'significant'
                self.graph_nx.nodes[stid]['weight'] = 0.0

    def save_name_to_id(self):
        file_path = 'data/emb/info/name_to_id.txt'
        os.makedirs(os.path.dirname(file_path), exist_ok=True)

        with open(file_path, 'w') as f:
            for name, id in self.name_to_id.items():
                f.write(f"{name}: {id}\n")

    def save_sorted_stids(self):
        file_path = 'data/emb/info/sorted_stids.txt'
        os.makedirs(os.path.dirname(file_path), exist_ok=True)

        stids = sorted(self.pathway_info.keys())

        with open(file_path, 'w') as f:
            for stid in stids:
                f.write(f"{stid}\n")

    def set_name_to_id(self):
        name_to_id = {}

        for id, info in self.pathway_info.items():
            name_to_id[info.name] = id

        return name_to_id


## `marker.py` — `Marker`

Runs Reactome over-representation enrichment analysis (via `reactome2py`) on a gene marker list, loading a local pathway→gene mapping as a fallback for `hit_genes` when Reactome's own `exp` field is empty, and writing both a simple and an expanded results CSV.


In [4]:
class Marker:
    def __init__(self, marker_list, p_value,
                 pathways_mapping_file="../data/processed/pathways_mapped_all_genes.tsv",
                 save_dir="reactome_embedding/results/enrichment"):
        self.marker_list = marker_list
        self.markers = set(marker_list)
        self.p_value = p_value
        self.save_dir = save_dir

        os.makedirs(save_dir, exist_ok=True)

        # Load pathway → genes mapping
        self.pathway_to_genes = self.load_pathway_mapping(pathways_mapping_file)

        # Run enrichment
        self.result, self.pathway_stats = self.enrichment_analysis()

        # Save CSVs
        self.save_results_csv()

    def load_pathway_mapping(self, mapping_file):
        df = pd.read_csv(mapping_file, sep="\t", low_memory=False)
        mapping = {}
        for _, row in df.iterrows():
            genes = [g for g in row[1:] if pd.notna(g)]
            mapping[row["PathwayID"]] = genes
        return mapping

    def enrichment_analysis(self):
        # Run Reactome analysis
        result = analysis.identifiers(
            ids=",".join(self.marker_list),
            interactors=False,
            page_size='1', page='1',
            species='Homo Sapiens',
            sort_by='ENTITIES_FDR',
            order='ASC',
            resource='TOTAL',
            p_value='1',
            include_disease=False,
            min_entities=None,
            max_entities=None,
            projection=True
        )
        token = result['summary']['token']
        token_result = analysis.token(
            token,
            species='Homo sapiens',
            page_size='-1',
            page='-1',
            sort_by='ENTITIES_FDR',
            order='ASC',
            resource='TOTAL',
            p_value='1',
            include_disease=False,
            min_entities=None,
            max_entities=None,
        )

        pathway_results = {}
        pathway_stats = {}

        for p in token_result['pathways']:
            stid = p['stId']
            name = p.get('name', "")
            entities = p['entities']
            p_val = float(entities.get('pValue', 1.0))
            fdr = entities.get('fdr')
            found = entities.get('found')
            total = entities.get('total')
            hit_ratio = entities.get('ratio')
            significance = 'significant' if p_val < self.p_value else 'non-significant'

            # Try Reactome exp first
            hit_genes = entities.get('exp', [])
            # Fallback: intersect input markers with local mapping
            if not hit_genes:
                hit_genes = list(self.markers.intersection(self.pathway_to_genes.get(stid, [])))
            hit_genes_str = ",".join(hit_genes)

            # Save full info
            pathway_results[stid] = {
                **p,
                "p_value": p_val,  # no rounding
                "significance": significance,
                "hit_genes": hit_genes_str
            }

            # Save stats for separate CSV
            pathway_stats[stid] = {
                "stId": stid,
                "name": name,
                "p_value": p_val,   # no rounding
                "fdr": fdr if fdr is not None else None,
                "found": found,
                "total": total,
                "hit_ratio": hit_ratio,
                "hit_genes": hit_genes_str,
                "significance": significance
            }

        return pathway_results, pathway_stats

    def save_results_csv(self):
        # Simple CSV (stId + p_value + significance)
        simple_csv = os.path.join(self.save_dir, "enrichment_simple.csv")
        simple_df = pd.DataFrame([
            {"stId": stid,
             "p_value": f"{info['p_value']:.6E}",
             "significance": info["significance"]}
            for stid, info in self.result.items()
        ])
        simple_df.to_csv(simple_csv, index=False)

        # Expanded CSV (all stats, with formatted p_value and fdr)
        expanded_csv = os.path.join(self.save_dir, "enrichment_expanded.csv")
        expanded_df = pd.DataFrame(self.pathway_stats.values())

        if "p_value" in expanded_df.columns:
            expanded_df["p_value"] = expanded_df["p_value"].apply(lambda x: f"{x:.6E}")
        if "fdr" in expanded_df.columns:
            expanded_df["fdr"] = expanded_df["fdr"].apply(lambda x: f"{x:.6E}" if pd.notna(x) else None)

        expanded_df.to_csv(expanded_csv, index=False)

        print(f"✅ Saved simple CSV: {simple_csv}")
        print(f"✅ Saved expanded CSV: {expanded_csv}")


## `model.py` — `GCNModel`

`GCNModel`: a stack of `dgl.nn.pytorch.GraphConv` layers (standard GCN, no attention/heads),
wrapped the same way the original `GATModel` was — an input linear projection from the scalar
`weight` node feature, then either raw node embeddings (`get_node_embeddings`) or a per-node
classification logit against `significance` (`forward`, when `do_train=True`).


In [5]:
from dgl.nn.pytorch import GraphConv


class GCNModel(nn.Module):
    def __init__(self,
                 in_feats,
                 out_feats,
                 feat_drop: float = 0.0,
                 activation: Optional[Callable] = None,
                 allow_zero_in_degree: bool = False,
                 bias: bool = True,
                 num_layers: int = 1,
                 do_train: bool = False) -> None:
        super(GCNModel, self).__init__()
        self.do_train = do_train
        self.linear = nn.Linear(1, in_feats)
        self.feat_drop = nn.Dropout(feat_drop)
        self.conv_0 = GraphConv(
            in_feats,
            out_feats,
            activation=activation,
            allow_zero_in_degree=allow_zero_in_degree,
            bias=bias,
        )
        self.relu = nn.LeakyReLU()
        self.layers = nn.ModuleList([
            GraphConv(
                out_feats,
                out_feats,
                activation=activation,
                allow_zero_in_degree=allow_zero_in_degree,
                bias=bias,
            )
            for _ in range(num_layers - 1)
        ])
        self.predict = nn.Linear(out_feats, 1)

    def forward(self, graph: DGLGraph) -> Tensor:
        weights = graph.ndata['weight'].unsqueeze(-1)
        features = self.linear(weights)
        features = self.feat_drop(features)
        graph = dgl.add_self_loop(graph)
        embedding = self.conv_0(graph, features)

        for conv in self.layers:
            embedding = self.relu(embedding)
            embedding = conv(graph, embedding)

        if not self.do_train:
            return embedding.detach()

        logits = self.predict(embedding).squeeze(-1)
        return logits

    def get_node_embeddings(self, graph: DGLGraph) -> Tensor:
        weights = graph.ndata['weight'].unsqueeze(-1)
        features = self.linear(weights)
        features = self.feat_drop(features)
        graph = dgl.add_self_loop(graph)
        embedding = self.conv_0(graph, features)

        for conv in self.layers:
            embedding = self.relu(embedding)
            embedding = conv(graph, embedding)

        return embedding


## `utils.py`

Helper functions tying `marker`/`network`/`dataset`/`model`/`train` together: building a Reactome network from a marker list, saving graphs to disk, and the two top-level pipeline entry points — `create_embedding_with_markers` (builds train/test pathway graphs from a gene list) and `create_embeddings` (trains or loads a `GCNModel` and produces per-graph embeddings).


In [6]:
def get_stid_mapping(graph):
    stid_mapping = {}  # Mapping of node_id to stId
    for node_id, data in graph.graph_nx.nodes(data=True):
        stId = data['stId']
        stid_mapping[node_id] = stId  # Store the mapping
    return stid_mapping  # Return the stId mapping

def create_network_from_markers(marker_list, p_value, kge):
    enrichment_analysis = marker.Marker(marker_list, p_value)
    graph = network.Network(enrichment_analysis.result, kge)
    return graph

def save_to_disk(graph, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    assert os.path.isdir(save_dir), 'Directory does not exist!'
    save_path = os.path.join(save_dir, graph.kge + '.pkl')
    pickle.dump(graph.graph_nx, open(save_path, 'wb'))


def create_embedding_with_markers(
    p_value=0.05,
    save=True,
    data_dir="reactome_embedding/data/emb"
):
    # ==========================================================
    # 1. Load marker symbols
    # ==========================================================
    csv_path = "../process/data/processed/gene_labels_driver_vs_nondriver.csv"

    data = pd.read_csv(csv_path)

    symbols = (
        data["gene"]
        .astype(str)
        .str.upper()
        .tolist()
    )

    print(f"✅ Marker symbols loaded: {len(symbols)}")

    # ==========================================================
    # 2. Train/test split
    # ==========================================================
    emb_train, emb_test = train_test_split(
        symbols,
        test_size=0.3,
        random_state=42
    )

    print(f"✅ Train genes: {len(emb_train)}")
    print(f"✅ Test genes: {len(emb_test)}")

    # ==========================================================
    # 3. Build networks
    # ==========================================================
    graph_test = create_network_from_markers(
        emb_test,
        p_value,
        "emb_test"
    )

    graph_train = create_network_from_markers(
        emb_train,
        p_value,
        "emb_train"
    )

    # ==========================================================
    # 4. Save
    # ==========================================================
    if save:
        os.makedirs(data_dir, exist_ok=True)

        save_dir = os.path.join(data_dir, "raw")
        os.makedirs(save_dir, exist_ok=True)

        save_to_disk(graph_train, save_dir)
        save_to_disk(graph_test, save_dir)

        print(f"💾 Saved graphs → {save_dir}")

    return graph_train, graph_test

def create_embeddings(load_model=True, save=True, data_dir='reactome_embedding/data/emb', hyperparams=None, plot=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data = dataset.PathwayDataset(data_dir)
    emb_dir = os.path.abspath(os.path.join(data_dir, 'embeddings'))
    if not os.path.isdir(emb_dir):
        os.mkdir(emb_dir)

    in_feats = hyperparams['in_feats']
    out_feats = hyperparams['out_feats']
    num_layers = hyperparams['num_layers']

    net = model.GCNModel(in_feats=in_feats, out_feats=out_feats, num_layers=num_layers).to(device)

    if load_model:
        model_path = os.path.abspath(os.path.join(data_dir, 'models/model.pth'))
        net.load_state_dict(torch.load(model_path))
    else:
        model_path = train.train(hyperparams=hyperparams, data_path=data_dir, plot=plot)
        net.load_state_dict(torch.load(model_path))

    embedding_dict = {}
    
    for idx in range(len(data)):
        graph, name = data[idx]
        graph = graph.to(device)  # Move graph to the same device as net
        
        with torch.no_grad():
            embedding = net(graph)
        embedding_dict[name] = embedding
        if save:
            emb_path = os.path.join(emb_dir, f'{name[:-4]}.pth')
            torch.save(embedding.cpu(), emb_path)

    return embedding_dict


## `train.py` — `FocalLoss` and the main `train()` loop

Trains `GCNModel` with a class-weighted focal loss on the binarized `significance` node label
(handling the strong class imbalance between "significant" and other pathway nodes), tracks
per-epoch loss/F1 on a single train graph and a single validation graph, checkpoints the best
model by validation loss, then clusters the resulting embeddings (KMeans) and dumps
heatmaps/PCA/t-SNE plots, per-cluster cancer-pathway gene lists, and a CSV of node embeddings.


In [7]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Ensure the input and target have the same shape
        if inputs.dim() > targets.dim():
            inputs = inputs.squeeze(dim=-1)
        elif targets.dim() > inputs.dim():
            targets = targets.squeeze(dim=-1)

        # Check if the shapes match after squeezing
        if inputs.size() != targets.size():
            raise ValueError(f"Target size ({targets.size()}) must be the same as input size ({inputs.size()})")

        BCE_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss

def train(hyperparams=None, data_path='reactome_embedding/data/emb', plot=True):
    num_epochs = hyperparams['num_epochs']
    in_feats = hyperparams['in_feats']
    out_feats = hyperparams['out_feats']
    num_layers = hyperparams['num_layers']
    learning_rate = hyperparams['lr']
    batch_size = hyperparams['batch_size']
    device = hyperparams['device']
    
    reactome_file_path = "reactome_embedding/data/NCBI2Reactome.csv"
    output_file_path = "reactome_embedding/data/NCBI_pathway_map.csv"
    gene_names_file_path = "reactome_embedding/data/gene_names.csv"
    pathway_map = create_pathway_map(reactome_file_path, output_file_path)
    gene_id_to_name_mapping, gene_id_to_symbol_mapping = read_gene_names(gene_names_file_path)
    
    model_path = os.path.join(data_path, 'models')
    model_path = os.path.join(model_path, f'model_dim{out_feats}_lay{num_layers}_epo{num_epochs}.pth')
    
    ds = dataset.PathwayDataset(data_path)
    ds_train = [ds[0]]
    ds_valid = [ds[1]]
    dl_train = GraphDataLoader(ds_train, batch_size=batch_size, shuffle=True)
    dl_valid = GraphDataLoader(ds_valid, batch_size=batch_size, shuffle=False)

    net = model.GCNModel(in_feats=in_feats, out_feats=out_feats, num_layers=num_layers, do_train=True).to(device)
    optimizer = optim.Adam(net.parameters(), lr=learning_rate)

    best_model = model.GCNModel(in_feats=in_feats, out_feats=out_feats, num_layers=num_layers, do_train=True)
    best_model.load_state_dict(copy.deepcopy(net.state_dict()))

    loss_per_epoch_train, loss_per_epoch_valid = [], []
    f1_per_epoch_train, f1_per_epoch_valid = [], []

    criterion = FocalLoss(alpha=1.0, gamma=2.0, reduction='mean')
    ##criterion = FocalLoss(alpha=0.25, gamma=2.0, reduction='mean')
    
    # Was [0.00001, 0.99999] -- that near-zero negative weight gave the
    # optimizer almost no signal from the majority class, which let it
    # collapse into a fixed, degenerate prediction set (F1 froze at a
    # constant value for hundreds of epochs). Softened here; try
    # [0.05, 0.95] too if this is still too aggressive/not aggressive enough.
    weight = torch.tensor([0.1, 0.9]).to(device)

    best_train_loss, best_valid_loss = float('inf'), float('inf')
    best_f1_score = 0.0

    max_f1_scores_train = []
    max_f1_scores_valid = []
    
    results_path = 'reactome_embedding/results/node_embeddings/'
    os.makedirs(results_path, exist_ok=True)

    all_embeddings_initial, cluster_labels_initial = calculate_cluster_labels(best_model, dl_train, device)
    all_embeddings_initial = all_embeddings_initial.reshape(all_embeddings_initial.shape[0], -1)  # Flatten 
    save_path_heatmap_initial= os.path.join(results_path, f'heatmap_stId_dim{out_feats}_lay{num_layers}_epo{num_epochs}_initial.png')
    save_path_matrix_initial= os.path.join(results_path, f'matrix_stId_dim{out_feats}_lay{num_layers}_epo{num_epochs}_initial.png')
    save_path_pca_initial = os.path.join(results_path, f'pca_dim{out_feats}_lay{num_layers}_epo{num_epochs}_initial.png')
    save_path_t_SNE_initial = os.path.join(results_path, f't-SNE_dim{out_feats}_lay{num_layers}_epo{num_epochs}_initial.png')
        
    for data in dl_train:
        graph, _ = data
        node_embeddings_initial= best_model.get_node_embeddings(graph).detach().cpu().numpy()
        graph_path = os.path.join(data_path, 'raw/emb_train.pkl')
        nx_graph = pickle.load(open(graph_path, 'rb'))

        assert len(cluster_labels_initial) == len(nx_graph.nodes), "Cluster labels and number of nodes must match"
        node_to_index_initial = {node: idx for idx, node in enumerate(nx_graph.nodes)}
        first_node_stId_in_cluster_initial= {}
        first_node_embedding_in_cluster_initial= {}

        stid_dic_initial= {}

        # Populate stid_dic with node stIds mapped to embeddings
        for node in nx_graph.nodes:
            stid_dic_initial[nx_graph.nodes[node]['stId']] = node_embeddings_initial[node_to_index_initial[node]]
            
        for node, cluster in zip(nx_graph.nodes, cluster_labels_initial):
            if cluster not in first_node_stId_in_cluster_initial:
                first_node_stId_in_cluster_initial[cluster] = nx_graph.nodes[node]['stId']
                first_node_embedding_in_cluster_initial[cluster] = node_embeddings_initial[node_to_index_initial[node]]

        print('first_node_stId_in_cluster_initial-------------------------------\n', first_node_stId_in_cluster_initial)
        stid_list = list(first_node_stId_in_cluster_initial.values())
        embedding_list_initial = list(first_node_embedding_in_cluster_initial.values())
        create_heatmap_with_stid(embedding_list_initial, stid_list, save_path_heatmap_initial)
        plot_cosine_similarity_matrix_for_clusters_with_values(embedding_list_initial, stid_list, save_path_matrix_initial)

        break

    visualize_embeddings_tsne(all_embeddings_initial, cluster_labels_initial, stid_list, save_path_t_SNE_initial)
    visualize_embeddings_pca(all_embeddings_initial, cluster_labels_initial, stid_list, save_path_pca_initial)
    silhouette_avg_ = silhouette_score(all_embeddings_initial, cluster_labels_initial)
    davies_bouldin_ = davies_bouldin_score(all_embeddings_initial, cluster_labels_initial)
    summary_  = f"Silhouette Score: {silhouette_avg_}\n"
    summary_ += f"Davies-Bouldin Index: {davies_bouldin_}\n"

    save_file_= os.path.join(results_path, f'dim{out_feats}_lay{num_layers}_epo{num_epochs}_initial.txt')
    with open(save_file_, 'w') as f:
        f.write(summary_)
      
    # Start training  
    with tqdm(total=num_epochs, desc="Training", unit="epoch", leave=False) as pbar:
        for epoch in range(num_epochs):
            loss_per_graph = []
            f1_per_graph = [] 
            net.train()
            for data in dl_train:
                graph, name = data
                name = name[0]
                logits = net(graph)
                labels = graph.ndata['significance'].unsqueeze(-1)
                weight_ = weight[labels.data.view(-1).long()].view_as(labels)

                loss = criterion(logits, labels)
                loss_weighted = loss * weight_
                loss_weighted = loss_weighted.mean()

                # Update parameters
                optimizer.zero_grad()
                loss_weighted.backward()
                optimizer.step()
                
                # Append output metrics
                loss_per_graph.append(loss_weighted.item())
                ##preds = (logits.sigmoid() > 0.5).squeeze(1).int()
                preds = (logits.sigmoid() > 0.5).int()
                labels = labels.squeeze(1).int()
                f1 = metrics.f1_score(labels, preds)
                f1_per_graph.append(f1)

            running_loss = np.array(loss_per_graph).mean()
            running_f1 = np.array(f1_per_graph).mean()
            loss_per_epoch_train.append(running_loss)
            f1_per_epoch_train.append(running_f1)

            # Validation iteration
            with torch.no_grad():
                loss_per_graph = []
                f1_per_graph = []
                net.eval()
                for data in dl_valid:
                    graph, name = data
                    name = name[0]
                    logits = net(graph)
                    labels = graph.ndata['significance'].unsqueeze(-1)
                    weight_ = weight[labels.data.view(-1).long()].view_as(labels)
                    loss = criterion(logits, labels)
                    loss_weighted = loss * weight_
                    loss_weighted = loss_weighted.mean()
                    loss_per_graph.append(loss_weighted.item())
                    ##preds = (logits.sigmoid() > 0.5).squeeze(1).int()
                    preds = (logits.sigmoid() > 0.5).int()
                    labels = labels.squeeze(1).int()
                    f1 = metrics.f1_score(labels, preds)
                    f1_per_graph.append(f1)

                running_loss = np.array(loss_per_graph).mean()
                running_f1 = np.array(f1_per_graph).mean()
                loss_per_epoch_valid.append(running_loss)
                f1_per_epoch_valid.append(running_f1)
                
                max_f1_train = max(f1_per_epoch_train)
                max_f1_valid = max(f1_per_epoch_valid)
                max_f1_scores_train.append(max_f1_train)
                max_f1_scores_valid.append(max_f1_valid)

                if running_loss < best_valid_loss:
                    best_train_loss = running_loss
                    best_valid_loss = running_loss
                    best_f1_score = running_f1
                    best_model.load_state_dict(copy.deepcopy(net.state_dict()))
                    print(f"Best F1 Score: {best_f1_score}")

            pbar.update(1)
            print(f"Epoch {epoch + 1} - Max F1 Train: {max_f1_train}, Max F1 Valid: {max_f1_valid}")

    all_embeddings, cluster_labels = calculate_cluster_labels(best_model, dl_train, device)
    all_embeddings = all_embeddings.reshape(all_embeddings.shape[0], -1)  # Flatten 
    print('cluster_labels=========================\n', cluster_labels)

    cos_sim = np.dot(all_embeddings, all_embeddings.T)
    norms = np.linalg.norm(all_embeddings, axis=1)
    cos_sim /= np.outer(norms, norms)

    if plot:
        loss_path = os.path.join(results_path, f'loss_dim{out_feats}_lay{num_layers}_epo{num_epochs}.png')
        f1_path = os.path.join(results_path, f'f1_dim{out_feats}_lay{num_layers}_epo{num_epochs}.png')
        max_f1_path = os.path.join(results_path, f'max_f1_dim{out_feats}_lay{num_layers}_epo{num_epochs}.png')
        matrix_path = os.path.join(results_path, f'matrix_dim{out_feats}_lay{num_layers}_epo{num_epochs}.png')
 
        draw_loss_plot(loss_per_epoch_train, loss_per_epoch_valid, loss_path)
        draw_max_f1_plot(max_f1_scores_train, max_f1_scores_valid, max_f1_path)
        draw_f1_plot(f1_per_epoch_train, f1_per_epoch_valid, f1_path)

    torch.save(best_model.state_dict(), model_path)

    save_path_pca = os.path.join(results_path, f'pca_dim{out_feats}_lay{num_layers}_epo{num_epochs}.png')
    save_path_t_SNE = os.path.join(results_path, f't-SNE_dim{out_feats}_lay{num_layers}_epo{num_epochs}.png')
    save_path_heatmap_= os.path.join(results_path, f'heatmap_stId_dim{out_feats}_lay{num_layers}_epo{num_epochs}.png')
    save_path_matrix = os.path.join(results_path, f'matrix_stId_dim{out_feats}_lay{num_layers}_epo{num_epochs}_.png')
    
    cluster_stId_dict = {}  # Dictionary to store clusters and corresponding stIds
    significant_stIds = []  # List to store significant stIds
    clusters_with_significant_stId = {}  # Dictionary to store clusters and corresponding significant stIds
    clusters_node_info = {}  # Dictionary to store node info for each cluster
    
    for data in dl_train:
        graph, _ = data
        node_embeddings = best_model.get_node_embeddings(graph).detach().cpu().numpy()
        graph_path = os.path.join(data_path, 'raw/emb_train.pkl')
        nx_graph = pickle.load(open(graph_path, 'rb'))

        assert len(cluster_labels) == len(nx_graph.nodes), "Cluster labels and number of nodes must match"
        node_to_index = {node: idx for idx, node in enumerate(nx_graph.nodes)}
        first_node_stId_in_cluster = {}
        first_node_embedding_in_cluster = {}

        stid_dic = {}

        # Populate stid_dic with node stIds mapped to embeddings
        for node in nx_graph.nodes:
            stid_dic[nx_graph.nodes[node]['stId']] = node_embeddings[node_to_index[node]]
            # Check if the node's significance is 'significant' and add its stId to the list
            
            ##print("node_embeddings[node_to_index[node]-------------------\n",graph.ndata['significance'][node_to_index[node]].item())
            if nx_graph.nodes[node]['significance'] == 'significant':
                significant_stIds.append(nx_graph.nodes[node]['stId'])
                ##print("significant_stIds-------------------\n",significant_stIds)
                ##print("nx_graph.nodes[node]['significance']-------------------\n",nx_graph.nodes[node]['significance'])
                
        for node, cluster in zip(nx_graph.nodes, cluster_labels):
            if cluster not in first_node_stId_in_cluster:
                first_node_stId_in_cluster[cluster] = nx_graph.nodes[node]['stId']
                first_node_embedding_in_cluster[cluster] = node_embeddings[node_to_index[node]]
                
            # Populate cluster_stId_dict
            if cluster not in cluster_stId_dict:
                cluster_stId_dict[cluster] = []
            cluster_stId_dict[cluster].append({
                    'stId': nx_graph.nodes[node]['stId'],
                    'name': nx_graph.nodes[node]['name']
                })

            # Populate clusters_with_significant_stId
            if cluster not in clusters_with_significant_stId:
                clusters_with_significant_stId[cluster] = []
            
            ##print('significant_stIds-------------------\n',significant_stIds)
            if nx_graph.nodes[node]['stId'] in significant_stIds:
                ##clusters_with_significant_stId[cluster].append(nx_graph.nodes[node]['stId'])
                clusters_with_significant_stId[cluster].append({
                    'stId': nx_graph.nodes[node]['stId'],
                    'name': nx_graph.nodes[node]['name']
                })
                    
            # Populate clusters_node_info with node information for each cluster
            if cluster not in clusters_node_info:
                clusters_node_info[cluster] = []
            node_info = {
                'stId': nx_graph.nodes[node]['stId'],
                'significance': graph.ndata['significance'][node_to_index[node]].item(),
                'other_info': nx_graph.nodes[node]  # Add other relevant info if necessary
            }
            clusters_node_info[cluster].append(node_info)
        
        print(first_node_stId_in_cluster)
        stid_list = list(first_node_stId_in_cluster.values())
        embedding_list = list(first_node_embedding_in_cluster.values())
        heatmap_data = pd.DataFrame(embedding_list, index=stid_list)
        create_heatmap_with_stid(embedding_list, stid_list, save_path_heatmap_)
        # Call the function to plot cosine similarity matrix for cluster representatives with similarity values
        plot_cosine_similarity_matrix_for_clusters_with_values(embedding_list, stid_list, save_path_matrix)

        break

    # === Save embeddings to CSV ===
    embedding_csv_path = os.path.join(
        results_path,
        f"embeddings_dim{out_feats}_lay{num_layers}_epo{num_epochs}.csv"
    )

    df_embeddings = pd.DataFrame.from_dict(
        stid_dic, 
        orient="index"
    ).reset_index()

    df_embeddings.rename(columns={"index": "stId"}, inplace=True)
    df_embeddings.to_csv(embedding_csv_path, index=False)

    print(f"✅ Node embeddings saved to {embedding_csv_path}")

    visualize_embeddings_tsne(all_embeddings, cluster_labels, stid_list, save_path_t_SNE)
    visualize_embeddings_pca(all_embeddings, cluster_labels, stid_list, save_path_pca)
    silhouette_avg = silhouette_score(all_embeddings, cluster_labels)
    davies_bouldin = davies_bouldin_score(all_embeddings, cluster_labels)

    print(f"Silhouette Score%%%%%%%%%%%%###########################: {silhouette_avg}")
    print(f"Davies-Bouldin Index: {davies_bouldin}")

    summary = f"Epoch {num_epochs} - Max F1 Train: {max_f1_train}, Max F1 Valid: {max_f1_valid}\n"
    ##summary += f"Best Train Loss: {best_train_loss}\n"
    summary += f"Best Validation Loss: {best_valid_loss}\n"
    summary += f"Best F1 Score: {max_f1_train}\n"
    summary += f"Best Val Score: {max_f1_valid}\n"
    summary += f"Silhouette Score: {silhouette_avg}\n"
    summary += f"Silhouette Score_ini: {silhouette_avg_}\n"
    summary += f"Davies-Bouldin Index: {davies_bouldin}\n"
    summary += f"Davies-Bouldin Index_ini: {davies_bouldin_}\n"

    save_file = os.path.join(results_path, f'dim{out_feats}_lay{num_layers}_epo{num_epochs}.txt')
    with open(save_file, 'w') as f:
        f.write(summary)

    
    
    graph_train, graph_test = utils.create_embedding_with_markers()  

    stid_mapping = utils.get_stid_mapping(graph_train)

    top_10_clusters = find_top_10_clusters_with_lowest_distance_and_connected_genes(all_embeddings, cluster_labels, cluster_stId_dict, pathway_map)

    output_file = os.path.join(results_path, f'significant_pathways_in_clusters_dim{out_feats}_lay{num_layers}_epo{num_epochs}.json')

    # Call the function with the top 10 clusters
    save_unique_genes_in_pathways(cluster_stId_dict, pathway_map, gene_id_to_name_mapping, gene_id_to_symbol_mapping, top_10_clusters, output_file)

    # Define file paths
    output_csv_file = "reactome_embedding/results/node_embeddings/unique_genes.csv"

    # Call the function
    save_unique_genes_to_csv(output_file, output_csv_file)

    ##min_distance_cluster_id = find_top_10_clusters_with_lowest_distance_and_connected_genes(all_embeddings, cluster_labels, clusters_with_significant_stId, pathway_map)

    # Call the function with only the selected cluster
    ##save_unique_genes_in_pathways(clusters_with_significant_stId, pathway_map, gene_id_to_name_mapping, gene_id_to_symbol_mapping, min_distance_cluster_id)

    return model_path


# def save_unique_genes_to_csv(json_file, output_csv_file):
#     """
#     Reads a JSON file with significant pathways and extracts unique genes associated with each pathway.
#     Saves the extracted gene information to a CSV file.

#     Parameters:
#         json_file (str): Path to the input JSON file.
#         output_csv_file (str): Path to the output CSV file.
#     """
#     with open(json_file, 'r') as f:
#         data = json.load(f)

#     unique_genes = set()
    
#     # Extract unique genes from pathways
#     for cluster_id, pathways in data.items():
#         for pathway in pathways:
#             genes = pathway.get("genes", [])
#             unique_genes.update(genes)

#     # Save unique genes to CSV
#     with open(output_csv_file, 'w', newline='') as csvfile:
#         writer = csv.writer(csvfile)
#         writer.writerow(["Gene Symbol"])  # Header row
#         for gene in unique_genes:
#             writer.writerow([gene])

#     print(f"Unique genes saved to {output_csv_file}")


### `train.py` helper functions

Clustering/pathway-lookup helpers, all the plotting utilities (`visualize_embeddings_pca`,
`visualize_embeddings_tsne`, loss/F1 curve plots, cosine-similarity + clustermap heatmaps), the
and the Reactome/gene-name file loaders (`create_pathway_map`, `read_gene_names`).


In [8]:
def save_unique_genes_to_csv(json_file, output_csv_file):
    import json, csv

    with open(json_file, 'r') as f:
        data = json.load(f)

    unique_genes = set()

    for cluster_id, pathways in data.items():
        for pathway in pathways:

            # CASE 1: pathway is dict
            if isinstance(pathway, dict):
                genes = pathway.get("genes", [])

            # CASE 2: pathway is string (ID)
            elif isinstance(pathway, str):
                genes = []  # or lookup from another mapping
                # e.g., genes = gene_mapping.get(pathway, [])

            else:
                genes = []

            unique_genes.update(genes)

    with open(output_csv_file, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Gene Symbol"])

        for gene in sorted(unique_genes):
            writer.writerow([gene])

    print(f"Unique genes saved to {output_csv_file}")

def find_top_10_clusters_with_lowest_distance_and_connected_genes(embeddings, labels, cluster_stId_dict, pathway_map):
    cluster_distances = {}
    valid_clusters = set()

    for cluster_id in set(labels):
        cluster_indices = [i for i, x in enumerate(labels) if x == cluster_id]
        cluster_embeddings = embeddings[cluster_indices]

        if len(cluster_embeddings) > 1:
            distance_matrix = pairwise_distances(cluster_embeddings)
            within_cluster_distance = distance_matrix.sum() / 2
            cluster_distances[cluster_id] = within_cluster_distance

            # Check if the cluster has connected genes
            stIds_names = cluster_stId_dict.get(cluster_id, [])

            ##print('stIds_names=============\n', stIds_names)

            
            stIds = [entry['stId'] for entry in stIds_names]
            ##print('stIds-----------------------\n',stIds)
            names = [entry['name'] for entry in stIds_names]


            # Check for connected genes
            if isinstance(stIds, list):
                for single_stId in stIds:
                    genes = pathway_map.get(single_stId, [])
                    if genes:  # Check if there are connected genes
                        valid_clusters.add(cluster_id)
                        break
            else:
                genes = pathway_map.get(stIds, [])
                if genes:  # Check if there are connected genes
                    valid_clusters.add(cluster_id)

    # Filter out clusters without connected genes
    valid_cluster_distances = {cid: dist for cid, dist in cluster_distances.items() if cid in valid_clusters}

    if not valid_cluster_distances:
        raise ValueError("No valid clusters with connected genes found")

    # Get the top 10 clusters with the lowest distances
    top_10_clusters = sorted(valid_cluster_distances, key=valid_cluster_distances.get)[:10]
    return top_10_clusters

def save_unique_genes_in_pathways(cluster_stId_dict, pathway_map, gene_id_to_name_mapping, gene_id_to_symbol_mapping, top_clusters, output_file):
    # Gather significant pathways and their unique connected genes
    significant_pathways_in_clusters = {}

    for cluster_id in top_clusters:
        stIds_names = cluster_stId_dict.get(cluster_id, [])
        # stId_name_pairs = [(entry['stId'], entry['name'], entry['weight'], entry['significance']) for entry in stIds_names]
        stId_name_pairs = []

        for entry in stIds_names:
            if isinstance(entry, dict):
                stId_name_pairs.append((
                    entry.get('stId'),
                    entry.get('name'),
                    entry.get('weight', 1.0),
                    entry.get('significance', 'not-found')
                ))
            else:
                # tuple fallback
                stId_name_pairs.append((
                    entry[0],
                    entry[1],
                    1.0,
                    'not-found'
                ))
 
        pathway_details = {}

        for stId, name, weight, significance in stId_name_pairs:
            if 'cancer' in name.lower():  # Only consider pathways with 'cancer' in their name
                ##print('name=========================\n',name)
                connected_genes = []
                genes = pathway_map.get(stId, [])
                unique_genes = set()  # To store unique genes for the pathway

                for gene_id in genes:
                    gene_name = gene_id_to_name_mapping.get(gene_id, None)
                    gene_symbol = gene_id_to_symbol_mapping.get(gene_id, None)
                    if gene_name and gene_symbol:  # Ensure that both name and symbol exist
                        connected_genes.append({
                            "gene_id": gene_id,
                            "gene_name": gene_name,
                            "gene_symbol": gene_symbol
                        })
                        unique_genes.add(gene_symbol)

                if connected_genes:  # Only add pathways with connected genes
                    pathway_details[stId] = {
                        "name": name,
                        "weight": weight,
                        "significance": significance,
                        "connected_genes": connected_genes,
                        "unique_genes": list(unique_genes)
                    }

        if pathway_details:  # Only add clusters with pathway details
            significant_pathways_in_clusters[str(cluster_id)] = pathway_details

    # Save the significant pathways with connected unique genes to a JSON file
    with open(output_file, 'w') as f:
        json.dump(significant_pathways_in_clusters, f, indent=4)

def plot_cosine_similarity_matrix_for_clusters_with_values(embeddings, stids, save_path):
    cos_sim = np.dot(embeddings, np.array(embeddings).T)
    norms = np.linalg.norm(embeddings, axis=1)
    cos_sim /= np.outer(norms, norms)

    plt.figure(figsize=(10, 8))
    
    vmin = cos_sim.min()
    vmax = cos_sim.max()
    # Create the heatmap with a custom color bar
    ##sns.heatmap(data, cmap='cividis')
    ##sns.heatmap(data, cmap='Blues') 'Greens' sns.heatmap(data, cmap='Spectral') 'coolwarm') 'YlGnBu') viridis cubehelix inferno

    ax = sns.heatmap(cos_sim, cmap="Spectral", annot=True, fmt=".3f", annot_kws={"size": 6},
                     xticklabels=stids, yticklabels=stids,
                     cbar_kws={"shrink": 0.2, "aspect": 15, "ticks": [vmin, vmax]})

    # Highlight the diagonal squares with value 1 by setting their background color to black
    for i in range(len(stids)):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color='black', alpha=0.5, zorder=3))
        
    ax.xaxis.tick_top()  # Move x-axis labels to the top
    ax.xaxis.set_label_position('top')  # Set x-axis label position to top
    plt.xticks(rotation=-30, fontsize=8, ha='right')  # Rotate x-axis labels, set font size, and align to the right
    plt.yticks(fontsize=8)  # Set font size for y-axis labels

    # Set the title below the plot
    ax.text(x=0.5, y=-0.03, s="Pathway-pathway similarities", fontsize=12, ha='center', va='top', transform=ax.transAxes)

    plt.savefig(save_path)
    ##plt.show()
    plt.close()
    
def create_pathway_map(reactome_file, output_file):
    """
    Extracts gene IDs with the same pathway STID and saves them to a new CSV file.

    Parameters:
    reactome_file (str): Path to the NCBI2Reactome.csv file.
    output_file (str): Path to save the output CSV file.
    """
    pathway_map = {}  # Dictionary to store gene IDs for each pathway STID

    # Read the NCBI2Reactome.csv file and populate the pathway_map
    with open(reactome_file, 'r') as file:
        reader = csv.reader(file, delimiter='\t')
        for row in reader:
            gene_id = row[0]
            pathway_stid = row[1]
            pathway_map.setdefault(pathway_stid, []).append(gene_id)

    # Write the pathway_map to the output CSV file
    with open(output_file, 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Pathway STID", "Gene IDs"])  # Write header
        for pathway_stid, gene_ids in pathway_map.items():
            writer.writerow([pathway_stid, ",".join(gene_ids)])
    
    return pathway_map
        
def read_gene_names(file_path):
    """
    Reads the gene names from a CSV file and returns a dictionary mapping gene IDs to gene names.

    Parameters:
    file_path (str): Path to the gene names CSV file.

    Returns:
    dict: A dictionary mapping gene IDs to gene names.
    """
    gene_id_to_name_mapping = {}
    gene_id_to_symbol_mapping = {}

    # Read the gene names CSV file and populate the dictionary
    with open(file_path, 'r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            gene_id = row['NCBI_Gene_ID']
            gene_name = row['Name']
            gene_symbol = row['Approved symbol']
            gene_id_to_name_mapping[gene_id] = gene_name
            gene_id_to_symbol_mapping[gene_id] = gene_symbol

    return gene_id_to_name_mapping, gene_id_to_symbol_mapping

def create_heatmap_with_stid(embedding_list, stid_list, save_path):
    # Convert the embedding list to a DataFrame
    heatmap_data = pd.DataFrame(embedding_list, index=stid_list)
    
    # Create a clustermap
    ax = sns.clustermap(heatmap_data, cmap='tab20', standard_scale=1, figsize=(10, 10))
    # Set smaller font sizes for various elements
    ax.ax_heatmap.tick_params(axis='both', which='both', labelsize=8)  # Tick labels
    ax.ax_heatmap.set_xlabel(ax.ax_heatmap.get_xlabel(), fontsize=8)  # X-axis label
    ax.ax_heatmap.set_ylabel(ax.ax_heatmap.get_ylabel(), fontsize=8)  # Y-axis label
    ax.ax_heatmap.collections[0].colorbar.ax.tick_params(labelsize=8)  # Color bar labels
    
    # Save the clustermap to a file
    plt.savefig(save_path)

    plt.close()

def calculate_cluster_labels(net, dataloader, device, num_clusters=20):
    all_embeddings = []
    net.eval()
    with torch.no_grad():
        for data in dataloader:
            graph, _ = data
            embeddings = net.get_node_embeddings(graph.to(device))
            all_embeddings.append(embeddings)
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    
    # Use KMeans clustering to assign cluster labels
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(all_embeddings)
    return all_embeddings, cluster_labels

def visualize_embeddings_pca(embeddings, cluster_labels, stid_list, save_path):
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)

    plt.figure(figsize=(10, 10))  # Square figure

    # Set the style
    sns.set(style="whitegrid")

    # Define unique clusters and sort them
    unique_clusters = np.unique(cluster_labels)
    sorted_clusters = sorted(unique_clusters)  # Sort the clusters

    # Define a color palette
    palette = sns.color_palette("viridis", len(sorted_clusters))

    # Create a scatter plot with a continuous colormap
    for i, cluster in enumerate(sorted_clusters):
        cluster_points = embeddings_2d[cluster_labels == cluster]
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f'{stid_list[cluster]}', s=20, color=palette[i], edgecolor='k')

    # Add labels and title
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('PCA of Embeddings')

    # Customize the grid and background
    ax = plt.gca()
    ax.set_facecolor('#eae6f0')
    ax.grid(True, which='both', color='white', linestyle='-', linewidth=1.0, alpha=0.9)  # Light grid lines with low alpha for near invisibility

    # Ensure the plot is square
    ax.set_aspect('equal', adjustable='box')

    # Create a custom legend with dot shapes and stid labels
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8, label=stid_list[cluster]) for i, cluster in enumerate(sorted_clusters)]
    plt.legend(handles=handles, title='Label', bbox_to_anchor=(1.02, 0.5), loc='center left', borderaxespad=0., fontsize='small', handlelength=0.5, handletextpad=0.5)

    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
        
def visualize_embeddings_tsne(embeddings, cluster_labels, stid_list, save_path):
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings)

    plt.figure(figsize=(10, 10))  # Square figure

    # Set the style
    sns.set(style="whitegrid")

    # Define unique clusters and sort them
    unique_clusters = np.unique(cluster_labels)
    sorted_clusters = sorted(unique_clusters)  # Sort the clusters

    # Define a color palette
    palette = sns.color_palette("viridis", len(sorted_clusters))
    
    # Create a scatter plot with a continuous colormap
    for i, cluster in enumerate(sorted_clusters):
        cluster_points = embeddings_2d[cluster_labels == cluster]
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f'{stid_list[cluster]}', s=20, color=palette[i], edgecolor='k')

    # Add labels and title
    plt.xlabel('dim_1')
    plt.ylabel('dim_2')
    plt.title('T-SNE of Embeddings')

    # Customize the grid and background
    ax = plt.gca()
    ax.set_facecolor('#eae6f0')
    ax.grid(True, which='both', color='white', linestyle='-', linewidth=1.0, alpha=0.9)  # Light grid lines with low alpha for near invisibility

    # Ensure the plot is square
    ax.set_aspect('equal', adjustable='box')

    # Create a custom legend with dot shapes and stid labels
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8, label=stid_list[cluster]) for i, cluster in enumerate(sorted_clusters)]
    plt.legend(handles=handles, title='Label', bbox_to_anchor=(1.02, 0.5), loc='center left', borderaxespad=0., fontsize='small', handlelength=0.5, handletextpad=0.5)

    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

def draw_loss_plot(train_loss, valid_loss, save_path):
    plt.figure()
    plt.plot(train_loss, label='train')
    plt.plot(valid_loss, label='validation')
    plt.title('Loss over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Customize the grid and background
    ax = plt.gca()
    ax.set_facecolor('#eae6f0')
    ax.grid(True, which='both', color='white', linestyle='-', linewidth=1.0, alpha=0.9)  # Light grid lines with low alpha for near invisibility
    
    plt.savefig(f'{save_path}')
    plt.close()

def draw_max_f1_plot(max_train_f1, max_valid_f1, save_path):
    plt.figure()
    plt.plot(max_train_f1, label='train')
    plt.plot(max_valid_f1, label='validation')
    plt.title('Max F1-score over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('F1-score')
    plt.legend()
    plt.savefig(f'{save_path}')
    plt.close()

def draw_f1_plot(train_f1, valid_f1, save_path):
    plt.figure()
    plt.plot(train_f1, label='train')
    plt.plot(valid_f1, label='validation')
    plt.title('F1-score over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('F1-score')
    plt.legend()

    # Customize the grid and background
    ax = plt.gca()
    ax.set_facecolor('#eae6f0')
    ax.grid(True, which='both', color='white', linestyle='-', linewidth=1.0, alpha=0.9)  # Light grid lines with low alpha for near invisibility

    plt.savefig(f'{save_path}')
    plt.close()



## Wire up cross-module references

The original files call each other as `dataset.PathwayDataset(...)`, `network.Network(...)`,
`model.GCNModel(...)`, `utils.create_embeddings(...)`, `train.train(...)`, etc. Now that
everything lives in one notebook namespace, this cell creates lightweight module-like objects
with those names so all of that code above works **unmodified** — no need to hunt down and
rewrite every `module.function(...)` call by hand.

Note the `train` name collision: `train.py` defines a function called `train`, and other files
call it as `train.train(...)`. The line order below matters — it captures the function object
first, then rebinds the name `train` to point at the module wrapper.


In [9]:
dataset = types.ModuleType('dataset')
dataset.PathwayDataset = PathwayDataset

network = types.ModuleType('network')
network.Network = Network

marker = types.ModuleType('marker')
marker.Marker = Marker

model = types.ModuleType('model')
model.GCNModel = GCNModel

utils = types.ModuleType('utils')
utils.get_stid_mapping = get_stid_mapping
utils.create_network_from_markers = create_network_from_markers
utils.save_to_disk = save_to_disk
utils.create_embedding_with_markers = create_embedding_with_markers
utils.create_embeddings = create_embeddings

# capture the `train` function defined above before rebinding the name to a module wrapper
_train_fn = train
train = types.ModuleType('train')
train.train = _train_fn
train.FocalLoss = FocalLoss
train.save_unique_genes_to_csv = save_unique_genes_to_csv
train.find_top_10_clusters_with_lowest_distance_and_connected_genes = find_top_10_clusters_with_lowest_distance_and_connected_genes
train.save_unique_genes_in_pathways = save_unique_genes_in_pathways
train.plot_cosine_similarity_matrix_for_clusters_with_values = plot_cosine_similarity_matrix_for_clusters_with_values
train.create_pathway_map = create_pathway_map
train.read_gene_names = read_gene_names
train.create_heatmap_with_stid = create_heatmap_with_stid
train.calculate_cluster_labels = calculate_cluster_labels
train.visualize_embeddings_pca = visualize_embeddings_pca
train.visualize_embeddings_tsne = visualize_embeddings_tsne
train.draw_loss_plot = draw_loss_plot
train.draw_max_f1_plot = draw_max_f1_plot
train.draw_f1_plot = draw_f1_plot

print("Cross-module references wired up.")


Cross-module references wired up.


## `gnn_embedding.py` — end-to-end entry point

Builds the train/test pathway graphs from markers, trains (or loads) `GCNModel`, and saves the
resulting embeddings to disk. The original script used `argparse` for CLI args; converted to a
plain `SimpleNamespace` here since this is running in a notebook rather than from the command
line — same default values as the original.

**Bug fixed:** the `if __name__ == '__main__':` block in `train.py` called
`train(hyperparams=hyperparams)` with a `hyperparams` dict that was missing `'in_feats'` and
and `'num_heads'` — both of which the *original* GAT-based `train()` unconditionally read via
`hyperparams['in_feats']` / `hyperparams['num_heads']`. That would raise a `KeyError` immediately.
(`num_heads` has since been removed entirely, now that the model is a GCN rather than a GAT.)
The `args` defaults below
include both keys, matching `gnn_embedding.py`'s own argparse defaults.


In [10]:
from types import SimpleNamespace

args = SimpleNamespace(
    data_dir='reactome_embedding/data/emb',
    output_file='reactome_embedding/data/emb/embeddings.pkl',
    p_value=0.05,
    save=True,
    num_epochs=10,
    in_feats=2,
    out_feats=128,
    num_layers=4,
    batch_size=1,
    lr=5e-4,
    print_embeddings=False,
)

# Top-level stage progress bar. Note: the actual 1000-epoch training loop
# inside utils.create_embeddings() -> train.train() already has its own
# per-epoch tqdm bar (cell 16) -- this is a separate, coarser bar showing
# which of the 3 pipeline stages is currently running.
pipeline_pbar = tqdm(total=3, desc="Pipeline", unit="stage")

# --- Stage 1: build train/test pathway graphs from markers ---
pipeline_pbar.set_description("Building marker graphs")
graph_train, graph_test = utils.create_embedding_with_markers(
    p_value=args.p_value,
    save=args.save,
    data_dir=args.data_dir
)
pipeline_pbar.update(1)

hyperparameters = {
    'num_epochs': args.num_epochs,
    'in_feats': args.in_feats,
    'out_feats': args.out_feats,
    'num_layers': args.num_layers,
    'batch_size': args.batch_size,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'lr': args.lr,
}

# --- Stage 2: train (this is where most of the time goes; watch the
# per-epoch bar from cell 16 for fine-grained progress) ---
pipeline_pbar.set_description("Training model")
embedding_dict = utils.create_embeddings(
    data_dir=args.data_dir,
    load_model=False,
    hyperparams=hyperparameters
)
pipeline_pbar.update(1)

if args.print_embeddings:
    print(embedding_dict)

# --- Stage 3: save embeddings to disk ---
pipeline_pbar.set_description("Saving embeddings")
with open(args.output_file, 'wb') as f:
    pickle.dump(embedding_dict, f)
pipeline_pbar.update(1)
pipeline_pbar.close()

print(f"Embeddings saved to {args.output_file}")


Pipeline:   0%|          | 0/3 [00:00<?, ?stage/s]

✅ Marker symbols loaded: 17506
✅ Train genes: 12254
✅ Test genes: 5252
✅ Saved simple CSV: reactome_embedding/results/enrichment/enrichment_simple.csv
✅ Saved expanded CSV: reactome_embedding/results/enrichment/enrichment_expanded.csv
✅ Saved simple CSV: reactome_embedding/results/enrichment/enrichment_simple.csv
✅ Saved expanded CSV: reactome_embedding/results/enrichment/enrichment_expanded.csv
💾 Saved graphs → reactome_embedding/data/emb/raw
first_node_stId_in_cluster_initial-------------------------------
 {4: 'R-HSA-9612973', 2: 'R-HSA-5683925', 8: 'R-HSA-1632843', 17: 'R-HSA-1632857', 13: 'R-HSA-5679239', 18: 'R-HSA-5672012', 12: 'R-HSA-5676229', 0: 'R-HSA-5682010', 9: 'R-HSA-5683588', 7: 'R-HSA-5682690', 19: 'R-HSA-5681987', 14: 'R-HSA-5205649', 5: 'R-HSA-9824888', 3: 'R-HSA-8948143', 10: 'R-HSA-8948139', 6: 'R-HSA-264458', 1: 'R-HSA-75035', 16: 'R-HSA-157906', 15: 'R-HSA-8961961', 11: 'R-HSA-8854051'}


Training:   0%|          | 0/10 [00:00<?, ?epoch/s]

Best F1 Score: 0.0005346735817783243
Epoch 1 - Max F1 Train: 0.000315905860053704, Max F1 Valid: 0.0005346735817783243
Best F1 Score: 0.0009260116677470136
Epoch 2 - Max F1 Train: 0.00032076984763432237, Max F1 Valid: 0.0009260116677470136
Best F1 Score: 0.0038022813688212928
Epoch 3 - Max F1 Train: 0.0005471457231442641, Max F1 Valid: 0.0038022813688212928
Best F1 Score: 0.005243838489774515
Epoch 4 - Max F1 Train: 0.002268431001890359, Max F1 Valid: 0.005243838489774515
Best F1 Score: 0.006617038875103391
Epoch 5 - Max F1 Train: 0.003121748178980229, Max F1 Valid: 0.006617038875103391
Best F1 Score: 0.013136288998357963
Epoch 6 - Max F1 Train: 0.004918032786885246, Max F1 Valid: 0.013136288998357963
Best F1 Score: 0.00909090909090909
Epoch 7 - Max F1 Train: 0.009917355371900827, Max F1 Valid: 0.013136288998357963
Best F1 Score: 0.0
Epoch 8 - Max F1 Train: 0.01818181818181818, Max F1 Valid: 0.013136288998357963
Best F1 Score: 0.0
Epoch 9 - Max F1 Train: 0.08333333333333333, Max F1 Val